## Connecting postgres database to python ##
psycopg2 is a python adapter that allows you to connect your postgresSQL sdatabase to your python. From python you can then manage connections of your database(closing and opening connection), run SQL querries directly, convert python datatypes eg lists, tuples, dictionaries into postgreSQL compatible formats and viceversa.

In [2]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import psycopg2

In [4]:
conn = psycopg2.connect(
    host='localhost',
    database='postgres',
    user='postgres',
    password='12345',
    port='5432'
)



In [5]:
sales = pd.read_sql("select * from retail.sales_transactions_production", conn)
sales.head()

C:\Users\ahmed\AppData\Local\Temp\ipykernel_17224\3064371451.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales = pd.read_sql("select * from retail.sales_transactions_production", conn)


,product_id,product_name,category,units_sold,unit_price,promotion_flag,stock_level_end_of_day,stockout_flag,date
0,SKU025,Sukuma Wiki Bunch,Produce,74,43.39,False,392.6,False,2024-03-03
1,SKU014,Cheese Slices,Dairy,74,112.46,False,563.7,False,2025-11-24
2,SKU006,Potato Crisps,Snacks,10,133.19,False,21.4,False,2024-06-05
3,SKU004,Energy Drink 250ml,Beverages,30,143.04,False,365.3,False,2025-09-29
4,SKU006,Potato Crisps,Snacks,20,133.19,False,65.4,False,2024-01-28


In [6]:

products = pd.read_sql('select * from retail.products_production',conn)

C:\Users\ahmed\AppData\Local\Temp\ipykernel_17224\1784742179.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql('select * from retail.products_production',conn)


In [7]:
products.head()

,product_id,product_name,category,unit_cost,unit_price,lead_time_days,safety_stock_days
0,SKU001,Soda 500ml,Beverages,126.94,154.61,5,4
1,SKU002,Bottled Water 1L,Beverages,119.13,96.01,6,4
2,SKU003,Juice 1L,Beverages,125.64,213.63,6,2
3,SKU004,Energy Drink 250ml,Beverages,93.94,143.04,3,4
4,SKU005,Iced Tea 500ml,Beverages,113.67,219.87,6,3


## merging the two tables into 1 flat table for EDA  ##

In [8]:
type(sales)

pandas.core.frame.DataFrame

In [9]:
type(products)

pandas.core.frame.DataFrame

In [10]:
mergetable = pd.merge(sales, products, on = "product_id", how = 'left')
mergetable.head()
mergetable.to_csv(r"C:\Users\ahmed\Documents\Data Science and AI(LUX)\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\cleaned_sales.csv",index=False)

In [11]:
mergetable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18275 entries, 0 to 18274
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   product_id              18275 non-null  object 
 1   product_name_x          18275 non-null  object 
 2   category_x              18275 non-null  object 
 3   units_sold              18275 non-null  int64  
 4   unit_price_x            18275 non-null  float64
 5   promotion_flag          18275 non-null  object 
 6   stock_level_end_of_day  18275 non-null  float64
 7   stockout_flag           18275 non-null  object 
 8   date                    18275 non-null  object 
 9   product_name_y          18275 non-null  object 
 10  category_y              18275 non-null  object 
 11  unit_cost               18275 non-null  float64
 12  unit_price_y            18275 non-null  float64
 13  lead_time_days          18275 non-null  int64  
 14  safety_stock_days       18275 non-null

## Feature engineering ##
enriching our data with features that will be used in EDA and the forecasting model

In [12]:

#changing date from 'object' to 'date'
mergetable['date']= pd.to_datetime(mergetable['date'])

#sorting my data first by product_id, then date
mergetable = mergetable.sort_values(['product_id','date']).reset_index(drop=True)

mergetable.head()


,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,category_y,unit_cost,unit_price_y,lead_time_days,safety_stock_days
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,Beverages,126.94,154.61,5,4
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,Beverages,126.94,154.61,5,4
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,Beverages,126.94,154.61,5,4
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,Beverages,126.94,154.61,5,4
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,Beverages,126.94,154.61,5,4


In [13]:
#calender features, (extracting day of the week, month, year) from dates

mergetable['day_of_week'] = mergetable['date'].dt.day_name()        #monday, tuesday......
mergetable['is_weekend'] = mergetable['date'].dt.dayofweek>=5       #'True' if Saturdays and Sundays, otherwise 'False' [Monday(0), Tuesday(1), Wed(2)....]
mergetable['Month'] = mergetable['date'].dt.month                   #'Jan', 'Feb', 'March'
mergetable['Quarter'] = mergetable['date'].dt.quarter
mergetable['is_festive'] = mergetable['Month'].isin([11,12])     #Nov and Dec, festive seasons
mergetable['day_of_year'] = mergetable['date'].dt.dayofyear


In [14]:
#Revenue
mergetable['Revenue'] = mergetable['unit_price_x'] * mergetable['units_sold']

In [15]:
mergetable.head()

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,unit_price_y,lead_time_days,safety_stock_days,day_of_week,is_weekend,Month,Quarter,is_festive,day_of_year,Revenue
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,154.61,5,4,Monday,False,1,1,False,1,4483.69
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,154.61,5,4,Tuesday,False,1,1,False,2,4174.47
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,154.61,5,4,Wednesday,False,1,1,False,3,3865.25
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,154.61,5,4,Thursday,False,1,1,False,4,5102.13
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,154.61,5,4,Friday,False,1,1,False,5,4019.86


In [16]:
#time_series feature engineering


#group the data by products, each product having a list of daily units sold 
g = mergetable.groupby('product_id')['units_sold'] 
mergetable['rolling_7d_avg'] = g.transform(lambda x: x.rolling(7, min_periods = 1).mean())
mergetable['rolling_28d_avg'] = g.transform(lambda x: x.rolling(28, min_periods = 1).mean())

#laging data(previous sales side by side with the current ones)
mergetable['previous_day_sale'] = g.shift(1)
mergetable['previous_week_sale'] = g.shift(7)

In [17]:
mergetable.head()

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,is_weekend,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,False,1,1,False,1,4483.69,29.0,29.0,NaN,NaN
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,False,1,1,False,2,4174.47,28.0,28.0,29.0,NaN
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,False,1,1,False,3,3865.25,27.0,27.0,27.0,NaN
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,False,1,1,False,4,5102.13,28.5,28.5,25.0,NaN
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,False,1,1,False,5,4019.86,28.0,28.0,33.0,NaN


In [18]:
mergetable[mergetable['product_id'] == 'SKU002']

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,is_weekend,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale
731,SKU002,Bottled Water 1l,Beverages,26,96.01,False,272.3,False,2024-01-01,Bottled Water 1L,...,False,1,1,False,1,2496.26,26.000000,26.000000,NaN,NaN
732,SKU002,Bottled Water 1l,Beverages,23,96.01,False,249.3,False,2024-01-02,Bottled Water 1L,...,False,1,1,False,2,2208.23,24.500000,24.500000,26.0,NaN
733,SKU002,Bottled Water 1l,Beverages,26,96.01,False,223.3,False,2024-01-03,Bottled Water 1L,...,False,1,1,False,3,2496.26,25.000000,25.000000,23.0,NaN
734,SKU002,Bottled Water 1l,Beverages,32,96.01,False,191.3,False,2024-01-04,Bottled Water 1L,...,False,1,1,False,4,3072.32,26.750000,26.750000,26.0,NaN
735,SKU002,Bottled Water 1l,Beverages,27,96.01,False,164.3,False,2024-01-05,Bottled Water 1L,...,False,1,1,False,5,2592.27,26.800000,26.800000,32.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1457,SKU002,Bottled Water 1l,Beverages,45,96.01,False,146.7,False,2025-12-27,Bottled Water 1L,...,True,12,4,True,361,4320.45,48.428571,50.964286,41.0,57.0
1458,SKU002,Bottled Water 1l,Beverages,59,96.01,False,87.7,False,2025-12-28,Bottled Water 1L,...,True,12,4,True,362,5664.59,49.428571,50.964286,45.0,52.0
1459,SKU002,Bottled Water 1l,Beverages,49,96.01,False,38.7,False,2025-12-29,Bottled Water 1L,...,False,12,4,True,363,4704.49,49.000000,51.428571,59.0,52.0
1460,SKU002,Bottled Water 1l,Beverages,48,96.01,False,659.6,False,2025-12-30,Bottled Water 1L,...,False,12,4,True,364,4608.48,51.428571,51.571429,49.0,31.0


In [19]:
#calculating days since the last promotion , (note in the begining if there wereno promotions, i want the results to be NaN)

def days_since_last_promo(sales):
    list_out = []
    last_promo_index = None
    for i, is_promo in enumerate(sales):
        if is_promo:
            last_promo_index = i
            days_since_last = 0
        elif last_promo_index is not None:
            days_since_last = i - last_promo_index
        else:
            days_since_last = None
        list_out.append(days_since_last)
    return list_out


y=mergetable.groupby('product_id')['promotion_flag']
mergetable['days_since_last_promo']= y.transform (lambda sales: days_since_last_promo(sales.tolist()))

    
            

In [20]:
mergetable.head()

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale,days_since_last_promo
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,1,1,False,1,4483.69,29.0,29.0,NaN,NaN,0
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,1,1,False,2,4174.47,28.0,28.0,29.0,NaN,0
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,1,1,False,3,3865.25,27.0,27.0,27.0,NaN,0
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,1,1,False,4,5102.13,28.5,28.5,25.0,NaN,0
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,1,1,False,5,4019.86,28.0,28.0,33.0,NaN,0


In [21]:
mergetable.to_csv(r"C:\Users\ahmed\Documents\Data Science and AI(LUX)\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\feature_engineered_sales.csv",index=False)

In [22]:
#loading my sales data after feature engineering
df = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science and AI(LUX)\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\feature_engineered_sales.csv")
df.head()

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale,days_since_last_promo
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,1,1,False,1,4483.69,29.0,29.0,NaN,NaN,0
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,1,1,False,2,4174.47,28.0,28.0,29.0,NaN,0
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,1,1,False,3,3865.25,27.0,27.0,27.0,NaN,0
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,1,1,False,4,5102.13,28.5,28.5,25.0,NaN,0
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,1,1,False,5,4019.86,28.0,28.0,33.0,NaN,0


In [23]:
#loading my cleaned sales data before feature engineering
df = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science and AI(LUX)\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\cleaned_sales.csv")
df.head()

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,category_y,unit_cost,unit_price_y,lead_time_days,safety_stock_days
0,SKU025,Sukuma Wiki Bunch,Produce,74,43.39,False,392.6,False,2024-03-03,Sukuma Wiki Bunch,Produce,63.20,43.39,8,2
1,SKU014,Cheese Slices,Dairy,74,112.46,False,563.7,False,2025-11-24,Cheese Slices,Dairy,84.23,112.46,3,4
2,SKU006,Potato Crisps,Snacks,10,133.19,False,21.4,False,2024-06-05,Potato Crisps,Snacks,50.45,133.19,9,2
3,SKU004,Energy Drink 250ml,Beverages,30,143.04,False,365.3,False,2025-09-29,Energy Drink 250ml,Beverages,93.94,143.04,3,4
4,SKU006,Potato Crisps,Snacks,20,133.19,False,65.4,False,2024-01-28,Potato Crisps,Snacks,50.45,133.19,9,2
